# Tech Challenge - Fase 1  
## Sistema de Apoio ao Diagnóstico de Câncer do Colo do Útero com Machine Learning

Este notebook apresenta uma solução inicial de IA aplicada à saúde da mulher, com foco em classificação de risco/diagnóstico a partir de dados estruturados. A variável alvo é `Biopsy`, que indica resultado positivo ou negativo de biópsia.

**Importante:** o modelo não substitui diagnóstico médico. Ele é apenas uma ferramenta de apoio à triagem e deve ser usado com supervisão clínica.

## 1. Instalação e importação das bibliotecas

In [ ]:
# Rode esta célula no Google Colab
!pip install -q shap joblib

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

pd.set_option('display.max_columns', 100)
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Carregamento dos dados

O notebook tenta carregar o arquivo localmente. Caso não exista, tenta baixar do repositório GitHub. Se o download falhar no Colab, faça upload manual do CSV para `/content/data/`.

In [ ]:
DATA_PATH = 'data/kag_risk_factors_cervical_cancer.csv'
RAW_URL = 'https://raw.githubusercontent.com/ChrisBelloni/Cervical-Cancer-Risk-Classification/main/data/kag_risk_factors_cervical_cancer.csv'

os.makedirs('data', exist_ok=True)

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    try:
        df = pd.read_csv(RAW_URL)
        df.to_csv(DATA_PATH, index=False)
    except Exception as e:
        raise FileNotFoundError(
            'Não foi possível carregar o dataset automaticamente. Faça upload do CSV para data/kag_risk_factors_cervical_cancer.csv'
        ) from e

df.head()

## 3. Entendimento inicial da base

Nesta etapa avaliamos dimensões, tipos de dados, amostras e presença de valores ausentes. No dataset original, valores ausentes podem aparecer como `?`, por isso eles serão convertidos para `NaN`.

In [ ]:
print('Dimensões:', df.shape)
display(df.head())
display(df.info())

df_clean = df.replace('?', np.nan).copy()
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

missing = df_clean.isna().mean().sort_values(ascending=False).reset_index()
missing.columns = ['variavel', 'percentual_ausente']
display(missing.head(15))

plt.figure(figsize=(10, 5))
sns.barplot(data=missing.head(15), x='percentual_ausente', y='variavel')
plt.title('Top 15 variáveis com maior percentual de valores ausentes')
plt.xlabel('Percentual de valores ausentes')
plt.ylabel('Variável')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/missing_values.png', dpi=150)
plt.show()

### Discussão da etapa

A base contém fatores clínicos e comportamentais relacionados ao risco de câncer do colo do útero. A presença de valores ausentes exige tratamento antes da modelagem. Como as variáveis são predominantemente numéricas ou binárias, foi adotada imputação pela mediana, que é robusta para distribuições assimétricas.

## 4. Análise exploratória da variável alvo

In [ ]:
TARGET = 'Biopsy'

target_counts = df_clean[TARGET].value_counts(dropna=False).reset_index()
target_counts.columns = [TARGET, 'quantidade']
display(target_counts)

plt.figure(figsize=(6,4))
sns.countplot(data=df_clean, x=TARGET)
plt.title('Distribuição da variável alvo: Biopsy')
plt.xlabel('Biopsy: 0 = negativo, 1 = positivo')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/target_distribution.png', dpi=150)
plt.show()

print('Proporção da classe positiva:')
print(df_clean[TARGET].value_counts(normalize=True))

### Discussão da variável alvo

O problema é sensível ao desbalanceamento de classes. Em saúde, um falso negativo pode ser mais grave que um falso positivo, pois uma paciente em risco poderia deixar de ser encaminhada para avaliação clínica. Por isso, a métrica de maior atenção será o **recall** da classe positiva, acompanhada de F1-score, precision, accuracy e ROC-AUC.

## 5. Estatísticas descritivas e distribuições

In [ ]:
display(df_clean.describe().T)

numeric_cols = df_clean.drop(columns=[TARGET]).columns[:12]
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df_clean[col], kde=True)
    plt.title(f'Distribuição de {col}')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/hist_{col.replace("/", "_").replace(" ", "_")}.png', dpi=120)
    plt.show()

## 6. Correlação entre variáveis

In [ ]:
corr = df_clean.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Mapa de correlação das variáveis')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/correlation_heatmap.png', dpi=150)
plt.show()

corr_target = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).head(15).reset_index()
corr_target.columns = ['variavel', 'correlacao_com_biopsy']
display(corr_target)

plt.figure(figsize=(8,5))
sns.barplot(data=corr_target, x='correlacao_com_biopsy', y='variavel')
plt.title('Top 15 correlações absolutas com Biopsy')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/target_correlations.png', dpi=150)
plt.show()

### Discussão da correlação

A correlação ajuda a identificar variáveis com maior relação linear com o resultado da biópsia. No entanto, correlação não implica causalidade. Por isso, ela é usada como apoio à compreensão dos dados, enquanto a decisão preditiva será avaliada com modelos supervisionados e métricas em dados não vistos.

## 7. Pré-processamento e divisão treino/validação/teste

Para evitar vazamento de informação, as variáveis `Hinselmann`, `Schiller` e `Citology` são removidas do conjunto preditivo, pois representam resultados diagnósticos auxiliares e poderiam tornar o problema artificialmente fácil. A divisão usada é:

- 60% treino
- 20% validação
- 20% teste

A validação é usada para escolher o melhor modelo. O teste é usado apenas na avaliação final.

In [ ]:
AUXILIARY_TARGETS = ['Hinselmann', 'Schiller', 'Citology']

X = df_clean.drop(columns=[TARGET] + [c for c in AUXILIARY_TARGETS if c in df_clean.columns])
y = df_clean[TARGET].astype(int)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print('Treino:', X_train.shape, y_train.value_counts(normalize=True).to_dict())
print('Validação:', X_val.shape, y_val.value_counts(normalize=True).to_dict())
print('Teste:', X_test.shape, y_test.value_counts(normalize=True).to_dict())

## 8. Modelagem

Serão testados quatro modelos de classificação:

1. Regressão Logística: modelo interpretável e bom baseline.
2. Árvore de Decisão: modelo simples, não linear e explicável.
3. Random Forest: conjunto de árvores, robusto para relações não lineares.
4. Gradient Boosting: modelo de boosting para capturar padrões complexos.

O pipeline aplica imputação de mediana e padronização antes do modelo.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
fitted_models = {}
reports = []

for name, model in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    pred = pipe.predict(X_val)
    proba = pipe.predict_proba(X_val)[:, 1]
    
    results.append({
        'modelo': name,
        'accuracy': accuracy_score(y_val, pred),
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'f1': f1_score(y_val, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, proba)
    })
    
    print('
===== ', name, '=====')
    print(classification_report(y_val, pred, zero_division=0))

metrics_val = pd.DataFrame(results).sort_values(['recall', 'f1', 'roc_auc'], ascending=False)
display(metrics_val)
metrics_val.to_csv(f'{OUTPUT_DIR}/metrics_validation.csv', index=False)

## 9. Validação cruzada

Além da validação holdout, é feita validação cruzada estratificada para verificar estabilidade dos resultados no conjunto de treino+validação.

In [ ]:
cv_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

for name, model in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    scores = cross_validate(pipe, X_trainval, y_trainval, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    row = {'modelo': name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    cv_results.append(row)

cv_metrics = pd.DataFrame(cv_results).sort_values(['recall_mean', 'f1_mean', 'roc_auc_mean'], ascending=False)
display(cv_metrics)
cv_metrics.to_csv(f'{OUTPUT_DIR}/metrics_cross_validation.csv', index=False)

## 10. Escolha do melhor modelo e avaliação final em teste

O melhor modelo será escolhido priorizando recall, seguido de F1-score e ROC-AUC. Essa escolha é adequada ao contexto médico, pois busca reduzir falsos negativos.

In [ ]:
best_name = metrics_val.iloc[0]['modelo']
best_model = fitted_models[best_name]
print('Melhor modelo pela validação:', best_name)

test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame([{
    'modelo': best_name,
    'accuracy': accuracy_score(y_test, test_pred),
    'precision': precision_score(y_test, test_pred, zero_division=0),
    'recall': recall_score(y_test, test_pred, zero_division=0),
    'f1': f1_score(y_test, test_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, test_proba)
}])

display(test_metrics)
print(classification_report(y_test, test_pred, zero_division=0))
test_metrics.to_csv(f'{OUTPUT_DIR}/metrics_test.csv', index=False)

ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title(f'Matriz de Confusão - Teste ({best_name})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_test.png', dpi=150)
plt.show()

RocCurveDisplay.from_predictions(y_test, test_proba)
plt.title(f'Curva ROC - Teste ({best_name})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curve_test.png', dpi=150)
plt.show()

## 11. Interpretabilidade: Feature Importance e SHAP

In [ ]:
model_step = best_model.named_steps['model']
imputer = best_model.named_steps['imputer']
scaler = best_model.named_steps['scaler']

X_train_processed = scaler.transform(imputer.transform(X_train))
X_test_processed = scaler.transform(imputer.transform(X_test))

if hasattr(model_step, 'feature_importances_'):
    importances = pd.DataFrame({
        'variavel': X.columns,
        'importancia': model_step.feature_importances_
    }).sort_values('importancia', ascending=False).head(15)
elif hasattr(model_step, 'coef_'):
    importances = pd.DataFrame({
        'variavel': X.columns,
        'importancia': np.abs(model_step.coef_[0])
    }).sort_values('importancia', ascending=False).head(15)
else:
    importances = pd.DataFrame(columns=['variavel', 'importancia'])

display(importances)

if not importances.empty:
    plt.figure(figsize=(8,5))
    sns.barplot(data=importances, x='importancia', y='variavel')
    plt.title(f'Principais variáveis - {best_name}')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/feature_importance.png', dpi=150)
    plt.show()

if SHAP_AVAILABLE:
    try:
        sample_size = min(100, X_test_processed.shape[0])
        X_sample = X_test_processed[:sample_size]
        if best_name in ['Random Forest', 'Gradient Boosting', 'Decision Tree']:
            explainer = shap.Explainer(model_step, X_train_processed)
            shap_values = explainer(X_sample)
        else:
            explainer = shap.Explainer(model_step, X_train_processed)
            shap_values = explainer(X_sample)
        shap.summary_plot(shap_values, X_sample, feature_names=X.columns, show=False)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/shap_summary.png', dpi=150)
        plt.show()
    except Exception as e:
        print('SHAP não foi executado neste ambiente. Feature importance já foi gerada.')
        print(e)

### Discussão da explicabilidade

A explicabilidade permite observar quais variáveis mais contribuíram para as previsões. Isso é essencial em saúde, pois profissionais clínicos precisam compreender os fatores que influenciam a triagem. Mesmo assim, importância estatística não deve ser interpretada automaticamente como causalidade clínica.

## 12. Salvamento do modelo e dos resultados

In [ ]:
joblib.dump(best_model, f'{OUTPUT_DIR}/best_model.joblib')
print('Modelo salvo em:', f'{OUTPUT_DIR}/best_model.joblib')
print('Arquivos gerados na pasta outputs:')
print(os.listdir(OUTPUT_DIR))

## 13. Conclusão crítica

A solução construída atende ao objetivo de criar uma base inicial de IA para apoio à triagem de risco em saúde da mulher. O modelo permite classificar pacientes com maior probabilidade de resultado positivo em biópsia a partir de fatores clínicos e comportamentais.

**Métrica principal:** recall, pois no contexto médico é mais grave deixar de identificar uma paciente em risco do que encaminhar uma paciente sem risco para avaliação adicional.

**Uso prático:** o modelo pode ser usado como apoio à priorização de triagem, sinalizando casos que merecem investigação clínica mais cuidadosa.

**Limitações:**

- Dataset limitado e possivelmente desbalanceado.
- Dados ausentes exigem imputação.
- O modelo não considera imagens, histórico clínico completo nem avaliação médica individualizada.
- Os resultados precisam de validação clínica antes de qualquer uso real.

**Responsabilidade:** a decisão final deve permanecer com profissionais de saúde. O sistema deve apoiar, não substituir, o julgamento médico.